# EEEM068 Action Recognition

Project authors: Prasanna Lamgade, Ben Davison, Chris Gainullin, Saba Ali, Youssef Abdelrahim.


In [ ]:
# !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
# !pip install -q transformers accelerate scikit-learn matplotlib seaborn numpy tqdm pyyaml pillow

import os
import random
import math
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import tqdm

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import TimesformerForVideoClassification, VideoMAEForVideoClassification

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Get the dataset

In [ ]:
!wget https://zenodo.org/records/7718549/files/HMDB_simp.zip
!unzip HMDB_simp.zip
!rm HMDB_simp.zip

--2026-03-19 17:32:29--  https://zenodo.org/records/7718549/files/HMDB_simp.zip
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 188.184.103.118, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2083847251 (1.9G) [application/octet-stream]
Saving to: ‘HMDB_simp.zip’

HMDB_simp.zip       100%[===================>]   1.94G  20.4MB/s    in 1m 42s  

2026-03-19 17:34:12 (19.5 MB/s) - ‘HMDB_simp.zip’ saved [2083847251/2083847251]

Archive:  HMDB_simp.zip
replace HMDB_simp/brush_hair/020E3BBA/0001.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

## Set paths

In [ ]:
DATA_DIR = "./HMDB_simp"
OUTPUT_DIR = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Config

In [ ]:
CFG = {
    "num_frames": 8,
    "resolution": 224,
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "seed": 42,
    "batch_size": 4,
    "num_workers": 0,
    "warmup_epochs": 3,
    "num_epochs": 6,
    "lr_head": 1e-3,
    "lr_backbone": 1e-5,
    "weight_decay": 1e-2,
    "grad_clip": 1.0,
    "early_stop_patience": 4,
}

CHECKPOINTS = {
    "timesformer": "facebook/timesformer-base-finetuned-k600",
    "videomae": "MCG-NJU/videomae-base-finetuned-kinetics",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## Utils

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


def count_parameters(model_inst) -> dict:
    total = 0
    for p in model_inst.parameters():
        total += p.numel()

    trainable = 0
    for p in model_inst.parameters():
        if p.requires_grad:
            trainable += p.numel()

    return {"total": total, "trainable": trainable}


## Dataset

In [ ]:
class HMDBDataset(Dataset):
    def __init__(self, root: str, num_frames: int = 8, transform=None):
        self.root = Path(root)
        self.num_frames = num_frames
        self.transform = transform
        self.samples = []
        self.class_to_idx = {}
        self.classes = []
        self._discover()

    def _discover(self):
        class_dirs = []
        self.classes = []
        for d in sorted(self.root.iterdir()):
            if d.is_dir():
                class_dirs.append(d)
                self.classes.append(d.name)

        self.class_to_idx = {}
        for i, c in enumerate(self.classes):
            self.class_to_idx[c] = i

        for class_dir in class_dirs:
            label = self.class_to_idx[class_dir.name]
            for sample_dir in sorted(class_dir.iterdir()):
                if sample_dir.is_dir():
                    self.samples.append((sample_dir, label))

        print(f"Found {len(self.classes)} classes, {len(self.samples)} samples")

    def _load_frames(self, sample_dir: Path):
        frame_paths = sorted(sample_dir.glob("*.jpg"))

        if len(frame_paths) == 0:
            raise ValueError(f"No files found in {sample_dir}")

        total = len(frame_paths)
        step = total / self.num_frames

        indices = []
        for i in range(self.num_frames):
            idx = int(i * step)
            idx = min(idx, total - 1)
            indices.append(idx)

        frames = []
        for idx in indices:
            img = Image.open(frame_paths[idx]).convert("RGB")
            frames.append(img)

        return frames

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        sample_dir, label = self.samples[i]
        pil_frames = self._load_frames(sample_dir)

        if self.transform:
            transformed = []
            for f in pil_frames:
                transformed.append(self.transform(f))
            frames = torch.stack(transformed)
        else:
            to_tensor = T.ToTensor()
            converted = []
            for f in pil_frames:
                converted.append(to_tensor(f))
            frames = torch.stack(converted)

        return frames, label


## Dataloader

In [ ]:
def get_transforms(resolution=224):
    return T.Compose([
        T.Resize((resolution, resolution)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def build_splits(dataset, train_ratio=0.70, val_ratio=0.15, seed=42):
    label_list = []
    for i in range(len(dataset)):
        label_list.append(dataset.samples[i][1])

    labels = np.array(label_list)
    indices = np.arange(len(dataset))

    test_ratio = 1.0 - train_ratio - val_ratio
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=seed)
    trainval_idx, test_idx = next(sss1.split(indices, labels))

    val_frac = val_ratio / (train_ratio + val_ratio)
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    train_idx, val_idx = next(sss2.split(trainval_idx, labels[trainval_idx]))

    train_idx = trainval_idx[train_idx]
    val_idx = trainval_idx[val_idx]

    print(f"Split sizes Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
    return train_idx.tolist(), val_idx.tolist(), test_idx.tolist()


def build_dataloaders(root, cfg):
    use_pin_memory = torch.cuda.is_available()
    dataset = HMDBDataset(
        root=root,
        num_frames=cfg["num_frames"],
        transform=get_transforms(cfg["resolution"]),
    )

    train_idx, val_idx, test_idx = build_splits(
        dataset,
        train_ratio=cfg["train_ratio"],
        val_ratio=cfg["val_ratio"],
        seed=cfg["seed"],
    )

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    test_subset = Subset(dataset, test_idx)

    train_loader = DataLoader(
        train_subset,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=cfg["num_workers"],
        pin_memory=use_pin_memory,
    )
    val_loader = DataLoader(
        val_subset,
        batch_size=cfg["batch_size"],
        shuffle=False,
        num_workers=cfg["num_workers"],
        pin_memory=use_pin_memory,
    )
    test_loader = DataLoader(
        test_subset,
        batch_size=cfg["batch_size"],
        shuffle=False,
        num_workers=cfg["num_workers"],
        pin_memory=use_pin_memory,
    )

    return train_loader, val_loader, test_loader


## Model loaders

In [ ]:
def load_timesformer(num_classes: int, checkpoint: str) -> TimesformerForVideoClassification:
    model = TimesformerForVideoClassification.from_pretrained(
        checkpoint,
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
        output_attentions=True,
    )
    return model

def load_videomae(num_classes: int, checkpoint: str) -> VideoMAEForVideoClassification:
    model_inst = VideoMAEForVideoClassification.from_pretrained(
        checkpoint,
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
        output_attentions=True,
    )
    return model_inst

## Metrics

In [ ]:
def _to_numpy_1d(values):
    arr = np.asarray(values)
    return arr.reshape(-1)


def compute_classification_metrics(y_true, y_pred, class_names=None):
    y_true = _to_numpy_1d(y_true).astype(int)
    y_pred = _to_numpy_1d(y_pred).astype(int)

    if y_true.shape[0] != y_pred.shape[0]:
        raise ValueError("y_true and y_pred must have the same length")

    if class_names is not None:
        labels = list(range(len(class_names)))
    else:
        labels = sorted(np.unique(np.concatenate([y_true, y_pred])).tolist())
        class_names = [str(v) for v in labels]

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels),
        "classification_report": report,
    }


def format_metrics_summary(metrics: dict) -> str:
    return (
        f"Acc: {metrics['accuracy']:.4f} | "
        f"Bal Acc: {metrics['balanced_accuracy']:.4f} | "
        f"P/R/F1 macro: {metrics['precision_macro']:.4f}/{metrics['recall_macro']:.4f}/{metrics['f1_macro']:.4f} | "
        f"F1 weighted: {metrics['f1_weighted']:.4f}"
    )


## Attention utilities

In [ ]:
@torch.no_grad()
def extract_attentions(model, frames, device):
    model.eval()
    frames = frames.to(device)
    out = model(pixel_values=frames, output_attentions=True)
    return out.attentions


def get_spatial_attention_map(attentions, layer_idx, num_frames, patch_grid=14):
    attn = attentions[layer_idx]
    attn = attn[0].mean(0)

    cls_attn = attn[0, 1:]

    num_patches = patch_grid * patch_grid
    total_tokens = cls_attn.shape[0]

    if total_tokens >= num_frames * num_patches:
        spatial = cls_attn[:num_frames * num_patches]
        spatial = spatial.reshape(num_frames, patch_grid, patch_grid)
        attn_map = spatial.mean(0).cpu().numpy()
    else:
        side = int(math.isqrt(total_tokens))
        attn_map = cls_attn[:side * side].reshape(side, side).cpu().numpy()

    attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
    return attn_map


def get_temporal_attention(attentions, layer_idx, num_frames, patch_grid=14):
    attn = attentions[layer_idx]
    attn = attn[0].mean(0)
    cls_attn = attn[0, 1:]

    total_tokens = cls_attn.shape[0]
    num_patches = patch_grid * patch_grid

    if total_tokens >= num_frames * num_patches:
        spatial = cls_attn[:num_frames * num_patches]
        spatial = spatial.reshape(num_frames, num_patches)
        temporal_weights = spatial.mean(1).cpu().numpy()
    elif total_tokens == num_frames:
        temporal_weights = cls_attn.cpu().numpy()
    else:
        chunk = total_tokens // num_frames
        if chunk > 0:
            temporal_weights = np.array([
                cls_attn[i * chunk:(i + 1) * chunk].mean().item()
                for i in range(num_frames)
            ])
        else:
            temporal_weights = np.ones(num_frames) / num_frames

    rng = temporal_weights.max() - temporal_weights.min()
    if rng > 1e-8:
        temporal_weights = (temporal_weights - temporal_weights.min()) / rng
    else:
        temporal_weights = np.ones(num_frames) / num_frames

    return temporal_weights


## Training loop

In [ ]:
class EarlyStopping:
    def __init__(self, patience=4, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = -np.inf
        self.counter = 0
        self.best_state = None

    def step(self, val_acc, model):
        if val_acc > self.best + self.min_delta:
            self.best = val_acc
            self.counter = 0
            self.best_state = {}
            for k, v in model.state_dict().items():
                self.best_state[k] = v.cpu().clone()
            print(f"New best val acc: {self.best:.4f}. Saving weights")
        else:
            self.counter += 1
        print(f"No improvement ({self.counter}/{self.patience})")
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)
            print(f"Restored best weights (val acc {self.best:.4f})")


def freeze_backbone(model):
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False

    trainable = 0
    for p in model.parameters():
        if p.requires_grad:
            trainable += p.numel()
    print(f"Backbone frozen\n Trainable params: {trainable:,} (head only)")


def unfreeze_all(model, lr_backbone, lr_head, weight_decay):
    for param in model.parameters():
        param.requires_grad = True

    trainable = 0
    for p in model.parameters():
        if p.requires_grad:
            trainable += p.numel()
    print(f"Backbone unfrozen\n Trainable params: {trainable:,}")

    backbone_params = []
    for name, param in model.named_parameters():
        if "classifier" not in name:
            backbone_params.append(param)

    head_params = []
    for name, param in model.named_parameters():
        if "classifier" in name:
            head_params.append(param)

    optimizer = optim.AdamW(
        [
            {"params": backbone_params, "lr": lr_backbone},
            {"params": head_params, "lr": lr_head},
        ],
        weight_decay=weight_decay,
    )

    return optimizer


def run_epoch(model, loader, optimizer, scheduler, grad_clip, device, train=True, return_predictions=False):
    if train:
        model.train()
    else:
        model.eval()

    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    num_batches = 0

    context = torch.enable_grad() if train else torch.no_grad()

    with context:
        for frames, labels in tqdm.tqdm(loader):
            num_batches += 1
            frames = frames.to(device)
            labels = labels.to(device)

            if train:
                optimizer.zero_grad()

            out = model(pixel_values=frames, labels=labels)
            loss = out.loss
            batch_size = frames.size(0)

            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if scheduler:
                    scheduler.step()

            predictions = out.logits.argmax(-1)
            correct += (predictions == labels).sum().item()
            total_loss += loss.item() * batch_size

            if return_predictions:
                all_preds.append(predictions.detach().cpu())
                all_labels.append(labels.detach().cpu())

            total += batch_size

    result = {
        "loss": total_loss / total,
        "accuracy": correct / total,
        "num_batches": num_batches,
    }

    if return_predictions:
        result["predictions"] = torch.cat(all_preds).numpy()
        result["labels"] = torch.cat(all_labels).numpy()

    return result


def _extract_class_names(loader):
    dataset_obj = loader.dataset
    if hasattr(dataset_obj, "dataset"):
        dataset_obj = dataset_obj.dataset
    if hasattr(dataset_obj, "classes"):
        return list(dataset_obj.classes)
    return None

from thop import profile

def _estimate_batch_flops(model, frames, labels, train=True):
    try:
        macs, params = profile(model, inputs=(frames,), verbose=False)
        fwd_flops = macs * 2

        if fwd_flops > 0:
            return fwd_flops * (3.0 if train else 1.0)
    except Exception as e:
        print(f"Error calculating FLOPs with thop: {e}")
        pass
    return None


def train(model, train_loader, val_loader, cfg, device, output_dir, model_arch):
    model = model.to(device)
    os.makedirs(output_dir, exist_ok=True)
    class_names = _extract_class_names(train_loader)

    freeze_backbone(model)

    trainable_params = []
    for p in model.parameters():
        if p.requires_grad:
            trainable_params.append(p)

    optimizer = optim.AdamW(
        trainable_params,
        lr=cfg["lr_head"],
        weight_decay=cfg["weight_decay"],
    )

    scheduler = None
    stopper = EarlyStopping(patience=cfg["early_stop_patience"])
    history = defaultdict(list)
    phase = 1

    train_batch_flops = None
    val_batch_flops = None

    try:
        train_frames, train_labels = next(iter(train_loader))
        train_batch_flops = _estimate_batch_flops(
            model,
            train_frames.to(device),
            train_labels.to(device),
            train=True,
        )
    except StopIteration:
        raise ValueError("train_loader is empty")

    try:
        val_frames, val_labels = next(iter(val_loader))
        val_batch_flops = _estimate_batch_flops(
            model,
            val_frames.to(device),
            val_labels.to(device),
            train=False,
        )
    except StopIteration:
        raise ValueError("val_loader is empty")

    for epoch in range(1, cfg["num_epochs"] + 1):
        if epoch == cfg["warmup_epochs"] + 1:
            optimizer = unfreeze_all(
                model,
                lr_backbone=cfg["lr_backbone"],
                lr_head=cfg["lr_head"],
                weight_decay=cfg["weight_decay"],
            )
            total_remaining = (cfg["num_epochs"] - cfg["warmup_epochs"]) * len(train_loader)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_remaining)
            phase = 2

        print(f"\nEpoch {epoch:02d}/{cfg['num_epochs']} [Phase {phase}]")

        train_stats = run_epoch(
            model, train_loader, optimizer, scheduler, cfg["grad_clip"], device, train=True
        )
        val_stats = run_epoch(
            model, val_loader, None, None, cfg["grad_clip"], device, train=False, return_predictions=True
        )

        val_metrics = compute_classification_metrics(
            y_true=val_stats["labels"],
            y_pred=val_stats["predictions"],
            class_names=class_names,
        )

        history["train_loss"].append(train_stats["loss"])
        history["train_acc"].append(train_stats["accuracy"])
        history["val_loss"].append(val_stats["loss"])
        history["val_acc"].append(val_stats["accuracy"])
        history["val_balanced_acc"].append(val_metrics["balanced_accuracy"])
        history["val_precision_macro"].append(val_metrics["precision_macro"])
        history["val_recall_macro"].append(val_metrics["recall_macro"])
        history["val_f1_macro"].append(val_metrics["f1_macro"])
        history["val_f1_weighted"].append(val_metrics["f1_weighted"])

        train_epoch_flops = None if train_batch_flops is None else train_batch_flops * train_stats["num_batches"]
        val_epoch_flops = None if val_batch_flops is None else val_batch_flops * val_stats["num_batches"]
        total_epoch_flops = None if (train_epoch_flops is None or val_epoch_flops is None) else (train_epoch_flops + val_epoch_flops)

        history["train_epoch_flops"].append(float("nan") if train_epoch_flops is None else float(train_epoch_flops))
        history["val_epoch_flops"].append(float("nan") if val_epoch_flops is None else float(val_epoch_flops))
        history["total_epoch_flops"].append(float("nan") if total_epoch_flops is None else float(total_epoch_flops))

        print(f"Train Loss: {train_stats['loss']:.4f} Acc: {train_stats['accuracy']:.4f}")
        print(f"Val Loss: {val_stats['loss']:.4f} Acc: {val_stats['accuracy']:.4f}")
        print(f"Val Metrics: {format_metrics_summary(val_metrics)}")

        if total_epoch_flops is None:
            print("Epoch FLOPs (estimate): unavailable")
        else:
            print(
                "Epoch FLOPs (estimate): "
                f"train={train_epoch_flops / 1e9:.2f} GFLOPs | "
                f"val={val_epoch_flops / 1e9:.2f} GFLOPs | "
                f"total={total_epoch_flops / 1e9:.2f} GFLOPs"
            )

        if stopper.step(val_stats["accuracy"], model):
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            break

    stopper.restore(model)

    final_val_stats = run_epoch(
        model, val_loader, None, None, cfg["grad_clip"], device, train=False, return_predictions=True
    )
    final_metrics = compute_classification_metrics(
        y_true=final_val_stats["labels"],
        y_pred=final_val_stats["predictions"],
        class_names=class_names,
    )

    checkpoint_path = os.path.join(output_dir, f"{model_arch}.pt")
    torch.save(model.state_dict(), checkpoint_path)
    print(f"\nSaved best checkpoint: {checkpoint_path}")

    metrics_dir = os.path.join(output_dir, "metrics")
    os.makedirs(metrics_dir, exist_ok=True)
    np.save(os.path.join(metrics_dir, f"{model_arch}_val_confusion_matrix.npy"), final_metrics["confusion_matrix"])
    with open(os.path.join(metrics_dir, f"{model_arch}_val_classification_report.json"), "w") as f:
        json.dump(final_metrics["classification_report"], f, indent=2)
    with open(os.path.join(metrics_dir, f"{model_arch}_val_metrics_summary.txt"), "w") as f:
        f.write(format_metrics_summary(final_metrics) + "\n")
    print(f"Saved classification metrics to: {metrics_dir}")

    return model, dict(history)

## Main experiment runner

In [ ]:
def run_experiment(model_name="timesformer", eval_only=False):
    if model_name not in ("timesformer", "videomae"):
        raise ValueError(f"Invalid model: {model_name}")

    cfg = dict(CFG)

    print(f"Model: {model_name}")
    print(f"Device: {DEVICE}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    seed_everything(cfg["seed"])

    if model_name == "timesformer":
        model = load_timesformer(num_classes=25, checkpoint=CHECKPOINTS[model_name])
    else:
        model = load_videomae(num_classes=25, checkpoint=CHECKPOINTS[model_name])
        # Videomae is fussy about these things
        cfg["num_frames"] = int(model.config.num_frames)
        image_size = model.config.image_size
        cfg["resolution"] = int(image_size[0] if isinstance(image_size, (tuple, list)) else image_size)

    train_loader, val_loader, test_loader = build_dataloaders(DATA_DIR, cfg)
    ds = HMDBDataset(root=DATA_DIR, num_frames=cfg["num_frames"])
    class_names = ds.classes

    params = count_parameters(model)
    print("\nParameters:")
    print(f"Total: {params['total']:,}")
    print(f"Trainable : {params['trainable']:,}")

    ckpt_path = os.path.join(OUTPUT_DIR, f"{model_name}.pt")

    if eval_only:
        if not os.path.exists(ckpt_path):
            raise FileNotFoundError(f"No checkpoint at {ckpt_path}")
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        model.to(DEVICE)
        print(f"Loaded checkpoint from {ckpt_path}")

        test_stats = run_epoch(
            model, test_loader, None, None, cfg["grad_clip"], DEVICE, train=False, return_predictions=True
        )
        test_metrics = compute_classification_metrics(
            y_true=test_stats["labels"],
            y_pred=test_stats["predictions"],
            class_names=class_names,
        )
        print("Test metrics:", format_metrics_summary(test_metrics))
        return model, {"test_metrics": test_metrics}

    model, history = train(
        model,
        train_loader,
        val_loader,
        cfg=cfg,
        device=DEVICE,
        output_dir=OUTPUT_DIR,
        model_arch=model_name,
    )

    print("\nTraining history:")
    for k, v in history.items():
        print(f"{k}: {[round(x, 4) for x in v]}")

    return model, history


## Run training or eval

In [ ]:
# Choose one: "timesformer" or "videomae"
MODEL_NAME = "videomae"
EVAL_ONLY = False

model, history = run_experiment(model_name=MODEL_NAME, eval_only=EVAL_ONLY)


Model: videomae
Device: cuda


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400, 768]) vs model:torch.Size([25, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400]) vs model:torch.Size([25])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


...

Train Loss: 0.0025 Acc: 1.0000
Val Loss: 0.1367 Acc: 0.9574
Val Metrics: Acc: 0.9574 | Bal Acc: 0.9579 | P/R/F1 macro: 0.9574/0.9579/0.9559 | F1 weighted: 0.9560
Epoch FLOPs (estimate): train=535315.57 GFLOPs | val=38295.03 GFLOPs | total=573610.59 GFLOPs
New best val acc: 0.9574. Saving weights
No improvement (0/4)

Epoch 06/6 [Phase 2]


100%|██████████| 47/47 [00:32<00:00,  1.44it/s]


Train Loss: 0.0002 Acc: 1.0000
Val Loss: 0.1346 Acc: 0.9574
Val Metrics: Acc: 0.9574 | Bal Acc: 0.9579 | P/R/F1 macro: 0.9574/0.9579/0.9559 | F1 weighted: 0.9560
Epoch FLOPs (estimate): train=535315.57 GFLOPs | val=38295.03 GFLOPs | total=573610.59 GFLOPs
No improvement (1/4)
Restored best weights (val acc 0.9574)


100%|██████████| 47/47 [00:33<00:00,  1.41it/s]



Saved best checkpoint: ./outputs/videomae.pt
Saved classification metrics to: ./outputs/metrics

Training history:
train_loss: [1.3923, 0.3312, 0.1586, 0.081, 0.0025, 0.0002]
train_acc: [0.7391, 0.9325, 0.9748, 0.9771, 1.0, 1.0]
val_loss: [0.4699, 0.2813, 0.2242, 0.154, 0.1367, 0.1346]
val_acc: [0.9096, 0.9362, 0.9468, 0.9468, 0.9574, 0.9574]
val_balanced_acc: [0.9093, 0.9371, 0.9479, 0.9471, 0.9579, 0.9579]
val_precision_macro: [0.9148, 0.9409, 0.9497, 0.9495, 0.9574, 0.9574]
val_recall_macro: [0.9093, 0.9371, 0.9479, 0.9471, 0.9579, 0.9579]
val_f1_macro: [0.9057, 0.9344, 0.9453, 0.9452, 0.9559, 0.9559]
val_f1_weighted: [0.9061, 0.9338, 0.945, 0.945, 0.956, 0.956]
train_epoch_flops: [535315565131776.0, 535315565131776.0, 535315565131776.0, 535315565131776.0, 535315565131776.0, 535315565131776.0]
val_epoch_flops: [38295025207296.0, 38295025207296.0, 38295025207296.0, 38295025207296.0, 38295025207296.0, 38295025207296.0]
total_epoch_flops: [573610590339072.0, 573610590339072.0, 5736105

## Attention check

In [ ]:
if MODEL_NAME == "videomae":
    # Videomae is fussy about these things
    CFG["num_frames"] = int(model.config.num_frames)
    image_size = model.config.image_size
    CFG["resolution"] = int(image_size[0] if isinstance(image_size, (tuple, list)) else image_size)

train_loader, _, _ = build_dataloaders(DATA_DIR, CFG)
frames, labels = next(iter(train_loader))
sample = frames[:1]  # one clip
attentions = extract_attentions(model.to(DEVICE), sample, DEVICE)
spatial = get_spatial_attention_map(attentions, layer_idx=-1, num_frames=sample.shape[1])
temporal = get_temporal_attention(attentions, layer_idx=-1, num_frames=sample.shape[1])
print("Spatial map shape:", spatial.shape)
print("Temporal weights:", temporal)


Found 25 classes, 1250 samples
Split sizes Train: 874 | Val: 188 | Test: 188
ImageClassifierOutput(loss=None, logits=tensor([[-1.3191, -1.9427,  0.1564, 14.8147,  0.1252, -0.8976, -1.2266,  6.0583,
         -0.6475, -1.1439,  0.2161,  0.5718, -2.9272, -3.1998, -3.0920,  0.4373,
         -0.0477, -1.0594,  0.0975, -2.2877, -0.3190, -2.3700, -2.8501,  1.2647,
         -2.6065]], device='cuda:0'), hidden_states=None, attentions=None)


TypeError: 'NoneType' object is not subscriptable

In [ ]:
!zip -r outputs.zip outputs/

  adding: outputs/ (stored 0%)
  adding: outputs/timesformer.pt (deflated 7%)
  adding: outputs/videomae.pt (deflated 7%)
  adding: outputs/metrics/ (stored 0%)
  adding: outputs/metrics/videomae_val_confusion_matrix.npy (deflated 97%)
  adding: outputs/metrics/timesformer_val_confusion_matrix.npy (deflated 96%)
  adding: outputs/metrics/videomae_val_metrics_summary.txt (deflated 30%)
  adding: outputs/metrics/timesformer_val_metrics_summary.txt (deflated 24%)
  adding: outputs/metrics/videomae_val_classification_report.json (deflated 86%)
  adding: outputs/metrics/timesformer_val_classification_report.json (deflated 85%)
